# Actividad 5 · Preparación de datos, entrenamiento y registro con MLflow
**MSCA2102 · Universidad Tecmilenio**

**Problema:** predecir si un cliente de telecomunicaciones abandona el servicio (*churn*), clasificación binaria.
**Dataset:** IBM Telco Customer Churn (7 043 clientes, 21 columnas), público.
**Modelos:** regresión logística vs. random forest, con `GridSearchCV` y validación cruzada estratificada de 5 pliegues.
**Registro:** MLflow (parámetros, métricas, artefactos, modelos y Model Registry).

**Cómo compilarlo en Google Colab:** `Entorno de ejecución → Ejecutar todo`. No requiere GPU ni credenciales.
El cuaderno crea la estructura `Actividad5/` completa (datos, fuentes, MLflow y resultados) y al final genera
`Actividad5.zip` para descargarla. Tiempo aproximado en Colab: 10–15 minutos.

## 0. Entorno

In [ ]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow==3.17.1"], check=True)

In [ ]:
import os, json, platform, warnings
from pathlib import Path

# Raíz del proyecto: /content/Actividad5 en Colab; la carpeta padre si se abre desde fuentes/
if EN_COLAB:
    RAIZ = Path("/content/Actividad5")
elif Path.cwd().name == "fuentes":
    RAIZ = Path.cwd().parent
else:
    RAIZ = Path.cwd() / "Actividad5"
for sub in ["datos/datos_ini", "datos/datos_limp", "fuentes", "resultados/figuras", "evidencias"]:
    (RAIZ / sub).mkdir(parents=True, exist_ok=True)
os.chdir(RAIZ)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore")
print("Raíz del proyecto:", RAIZ)

## 1. Código fuente del proyecto
Las dos celdas siguientes escriben `fuentes/datos_prep.py` (limpieza y versionado) y `fuentes/train.py`
(entrenamiento, ajuste y registro en MLflow). Es exactamente el mismo código que está en el repositorio;
el cuaderno lo importa para que notebook y scripts produzcan los mismos resultados.

In [ ]:
%%writefile fuentes/datos_prep.py
"""
datos_prep.py — Obtención, validación, limpieza y versionado del dataset
IBM Telco Customer Churn.

Uso desde la terminal (raíz de Actividad5/):
    python fuentes/datos_prep.py

Genera:
    datos/datos_ini/Telco-Customer-Churn.csv      (copia íntegra de la fuente)
    datos/datos_limp/telco_churn_limpio.csv       (dataset limpio, v1.1.0)
    datos/datos_limp/telco_churn_limpio.meta.json (versión, huellas SHA-256 y bitácora)

El escalado y la codificación one-hot NO se guardan en el CSV limpio: se
aplican dentro del Pipeline de scikit-learn (train.py) para que se ajusten
sólo con los datos de entrenamiento de cada pliegue y no haya fuga de
información.
"""
from __future__ import annotations

import hashlib
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Constantes del dataset
# ---------------------------------------------------------------------------
URL_FUENTE = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
NOMBRE_INI = "Telco-Customer-Churn.csv"
NOMBRE_LIMP = "telco_churn_limpio.csv"
VERSION_DATOS = "1.1.0"

ID = "customerID"
OBJETIVO = "Churn"

SERVICIOS_INTERNET = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
CATEGORICAS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", *SERVICIOS_INTERNET,
    "Contract", "PaperlessBilling", "PaymentMethod",
]
NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges", "n_servicios"]

# Dominio válido de cada columna categórica en los datos originales
DOMINIOS = {
    "gender": {"Female", "Male"},
    "SeniorCitizen": {0, 1},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "MultipleLines": {"Yes", "No", "No phone service"},
    "InternetService": {"DSL", "Fiber optic", "No"},
    **{c: {"Yes", "No", "No internet service"} for c in SERVICIOS_INTERNET},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "PaperlessBilling": {"Yes", "No"},
    "PaymentMethod": {
        "Electronic check", "Mailed check",
        "Bank transfer (automatic)", "Credit card (automatic)",
    },
    "Churn": {"Yes", "No"},
}

# Tolerancia para la regla de consistencia TotalCharges ≈ tenure × MonthlyCharges
TOLERANCIA_CARGOS = 0.25


# ---------------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------------
def sha256(ruta: str | Path) -> str:
    """Huella SHA-256 de un archivo (identifica la versión exacta de los datos)."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def descargar_datos(destino: str | Path, url: str = URL_FUENTE) -> Path:
    """Descarga el CSV original si no existe ya en `destino`."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        urllib.request.urlretrieve(url, destino)
    return destino


def cargar_datos_ini(ruta: str | Path) -> pd.DataFrame:
    """Lee el CSV original sin conversiones implícitas (todo como texto salvo lo numérico limpio)."""
    return pd.read_csv(ruta, dtype={ID: str, "TotalCharges": str})


# ---------------------------------------------------------------------------
# Diagnóstico de calidad
# ---------------------------------------------------------------------------
def perfil_calidad(df: pd.DataFrame) -> dict:
    """Resumen de calidad del dataset original: tipos, nulos, blancos,
    duplicados, valores fuera de dominio y desbalance de la variable objetivo."""
    tc_texto = df["TotalCharges"].astype(str).str.strip()
    tc = pd.to_numeric(tc_texto.replace("", np.nan), errors="coerce")
    sin_id = df.drop(columns=[ID])
    fuera_dominio = {}
    for col, dominio in DOMINIOS.items():
        n = int((~df[col].isin(dominio)).sum())
        if n:
            fuera_dominio[col] = n
    return {
        "filas": int(len(df)),
        "columnas": int(df.shape[1]),
        "nulos_explicitos": int(df.isna().sum().sum()),
        "totalcharges_en_blanco": int((tc_texto == "").sum()),
        "totalcharges_no_numerico": int(tc.isna().sum()),
        "ids_duplicados": int(df[ID].duplicated().sum()),
        "duplicados_exactos_sin_id": int(sin_id.duplicated().sum()),
        "duplicados_atributos_sin_objetivo": int(sin_id.drop(columns=[OBJETIVO]).duplicated().sum()),
        "valores_fuera_de_dominio": fuera_dominio,
        "tasa_churn": round(float((df[OBJETIVO] == "Yes").mean()), 4),
    }


# ---------------------------------------------------------------------------
# Limpieza y estandarización
# ---------------------------------------------------------------------------
def limpiar(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:
    """Aplica las reglas de limpieza en orden y devuelve (df_limpio, bitácora).

    Reglas:
      R1  Recortar espacios en todas las columnas de texto.
      R2  Validar dominios de las categóricas (falla si aparece un valor no esperado).
      R3  TotalCharges de texto a numérico; los blancos se convierten en nulos.
      R4  Regla de negocio: si tenure == 0 el cliente aún no tiene ciclo facturado,
          por lo que TotalCharges = 0 (imputación por regla, no por la mediana).
      R5  SeniorCitizen 0/1 -> "No"/"Yes" para tratarla igual que las demás binarias.
      R6  "No internet service" / "No phone service" -> "No": la ausencia del servicio
          base ya está en InternetService / PhoneService (evita colinealidad perfecta).
      R7  Eliminar duplicados exactos (todas las columnas salvo customerID), conservando
          el primero; evita que un mismo perfil quede en entrenamiento y prueba.
      R8  Marcar (sin eliminar) registros cuyo TotalCharges se aleja más de ±25 % de
          tenure × MonthlyCharges: pueden ser cambios de tarifa legítimos.
      R9  Variable derivada n_servicios (servicios adicionales contratados, 0-7).
      R10 Objetivo Churn "Yes"/"No" -> 1/0; tipos finales: category / int / float.
      R11 Eliminar customerID (identificador sin valor predictivo).
    """
    bitacora: list[dict] = []
    d = df.copy()

    def registrar(regla: str, descripcion: str, filas_afectadas: int) -> None:
        bitacora.append({"regla": regla, "descripcion": descripcion,
                         "filas_afectadas": int(filas_afectadas), "filas_resultantes": int(len(d))})

    # R1
    texto = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) in ("string", "str")]
    cambios = 0
    for c in texto:
        s = d[c].astype(str)
        cambios += int((s != s.str.strip()).sum())
        d[c] = s.str.strip()
    registrar("R1", "Recorte de espacios en columnas de texto", cambios)

    # R2
    for col, dominio in DOMINIOS.items():
        valores = d[col] if col != "SeniorCitizen" else d[col].astype(int)
        invalidos = set(valores.unique()) - dominio
        if invalidos:
            raise ValueError(f"Valores fuera de dominio en {col}: {invalidos}")
    registrar("R2", "Validación de dominios categóricos (sin valores inválidos)", 0)

    # R3
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"].replace("", np.nan), errors="coerce")
    nulos_tc = int(d["TotalCharges"].isna().sum())
    registrar("R3", "TotalCharges convertido a numérico; blancos -> nulo", nulos_tc)

    # R4
    mascara = d["TotalCharges"].isna() & (d["tenure"] == 0)
    d.loc[mascara, "TotalCharges"] = 0.0
    if d["TotalCharges"].isna().any():
        raise ValueError("Quedan nulos en TotalCharges con tenure > 0: revisar la fuente.")
    registrar("R4", "Regla de negocio: tenure = 0 => TotalCharges = 0", int(mascara.sum()))

    # R5
    d["SeniorCitizen"] = d["SeniorCitizen"].astype(int).map({0: "No", 1: "Yes"})
    registrar("R5", "SeniorCitizen 0/1 -> No/Yes", len(d))

    # R6
    n6 = 0
    for c in SERVICIOS_INTERNET + ["MultipleLines"]:
        m = d[c].isin(["No internet service", "No phone service"])
        n6 += int(m.sum())
        d.loc[m, c] = "No"
    registrar("R6", "'No internet/phone service' -> 'No' (7 columnas)", n6)

    # R7
    antes = len(d)
    d = d.loc[~d.drop(columns=[ID]).duplicated(keep="first")].reset_index(drop=True)
    registrar("R7", "Eliminación de duplicados exactos (sin considerar customerID)", antes - len(d))

    # R8
    esperado = d["tenure"] * d["MonthlyCharges"]
    razon = np.where(esperado > 0, d["TotalCharges"] / esperado.replace(0, np.nan), 1.0)
    d["anomalia_cargos"] = (np.abs(razon - 1) > TOLERANCIA_CARGOS).astype(int)
    registrar("R8", f"Marca de inconsistencia de cargos (|razón-1| > {TOLERANCIA_CARGOS})",
              int(d["anomalia_cargos"].sum()))

    # R9
    adicionales = SERVICIOS_INTERNET + ["MultipleLines"]
    d["n_servicios"] = (d[adicionales] == "Yes").sum(axis=1).astype(int)
    registrar("R9", "Variable derivada n_servicios (0-7)", len(d))

    # R10
    d[OBJETIVO] = (d[OBJETIVO] == "Yes").astype(int)
    for c in CATEGORICAS:
        d[c] = d[c].astype("category")
    d["tenure"] = d["tenure"].astype(int)
    d["MonthlyCharges"] = d["MonthlyCharges"].astype(float)
    d["TotalCharges"] = d["TotalCharges"].astype(float)
    registrar("R10", "Objetivo a 1/0 y tipos finales (category, int, float)", len(d))

    # R11
    d = d.drop(columns=[ID])
    registrar("R11", "Eliminación de customerID", 0)

    columnas = CATEGORICAS + NUMERICAS + ["anomalia_cargos", OBJETIVO]
    return d[columnas], bitacora


def validar_limpio(d: pd.DataFrame) -> None:
    """Pruebas mínimas de calidad sobre el dataset limpio (fallan con AssertionError)."""
    assert d.isna().sum().sum() == 0, "Hay nulos en el dataset limpio"
    assert d.duplicated().sum() == 0, "Hay duplicados exactos en el dataset limpio"
    assert set(d[OBJETIVO].unique()) <= {0, 1}, "Objetivo no binario"
    assert (d["tenure"] >= 0).all() and (d["MonthlyCharges"] > 0).all()
    assert (d["TotalCharges"] >= 0).all()
    for c in SERVICIOS_INTERNET + ["MultipleLines"]:
        assert set(d[c].astype(str).unique()) <= {"Yes", "No"}, f"Categoría residual en {c}"


# ---------------------------------------------------------------------------
# Versionado
# ---------------------------------------------------------------------------
def guardar_limpio(d: pd.DataFrame, bitacora: list[dict], ruta_ini: Path,
                   dir_limp: Path, perfil: dict) -> dict:
    """Guarda el CSV limpio y un archivo .meta.json con versión, huellas y bitácora."""
    dir_limp.mkdir(parents=True, exist_ok=True)
    ruta_csv = dir_limp / NOMBRE_LIMP
    d.to_csv(ruta_csv, index=False, lineterminator="\n")
    meta = {
        "dataset": "IBM Telco Customer Churn",
        "version": VERSION_DATOS,
        "fuente": URL_FUENTE,
        "archivo_ini": f"datos/datos_ini/{ruta_ini.name}",
        "sha256_ini": sha256(ruta_ini),
        "archivo_limpio": f"datos/datos_limp/{NOMBRE_LIMP}",
        "sha256_limpio": sha256(ruta_csv),
        "filas": int(len(d)),
        "columnas": list(d.columns),
        "tasa_churn": round(float(d[OBJETIVO].mean()), 4),
        "perfil_calidad_ini": perfil,
        "bitacora_limpieza": bitacora,
        "generado_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    with open(dir_limp / (NOMBRE_LIMP.replace(".csv", ".meta.json")), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    return meta


def cargar_limpio(ruta: str | Path) -> pd.DataFrame:
    """Lee el CSV limpio restaurando los tipos categóricos."""
    d = pd.read_csv(ruta)
    for c in CATEGORICAS:
        d[c] = d[c].astype(str).astype("category")
    return d


def preparar(raiz: str | Path = ".") -> dict:
    """Flujo completo: descarga (si hace falta) -> perfil -> limpieza -> validación -> versionado."""
    raiz = Path(raiz)
    ruta_ini = descargar_datos(raiz / "datos" / "datos_ini" / NOMBRE_INI)
    df = cargar_datos_ini(ruta_ini)
    perfil = perfil_calidad(df)
    limpio, bitacora = limpiar(df)
    validar_limpio(limpio)
    return guardar_limpio(limpio, bitacora, ruta_ini, raiz / "datos" / "datos_limp", perfil)


if __name__ == "__main__":
    m = preparar(Path(__file__).resolve().parents[1])
    print(f"Dataset limpio v{m['version']}: {m['filas']} filas · churn {m['tasa_churn']:.2%}")
    print(f"SHA-256 original: {m['sha256_ini']}")
    print(f"SHA-256 limpio:   {m['sha256_limpio']}")
    for paso in m["bitacora_limpieza"]:
        print(f"  {paso['regla']:>3}  {paso['descripcion']:<70} {paso['filas_afectadas']:>6}")

In [ ]:
%%writefile fuentes/train.py
"""
train.py — Entrenamiento, ajuste de hiperparámetros (GridSearchCV + k-fold
estratificado), evaluación, comparación estadística y registro en MLflow.

Uso desde la terminal (raíz de Actividad5/), después de datos_prep.py:
    python fuentes/train.py
    mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000

Experimento MLflow: "Actividad5-churn-telco"
    · 1 run de referencia (DummyClassifier)
    · 1 run padre por algoritmo con sus candidatos del grid como runs hijos
    · 1 run de comparación estadística entre los dos mejores modelos
Los modelos ganadores se registran en el Model Registry.
"""
from __future__ import annotations

import inspect
import json
import os
import platform
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow.models import infer_signature
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_predict, cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.insert(0, str(Path(__file__).resolve().parent))
import datos_prep as dp  # noqa: E402

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ill-defined.*")
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")

# ---------------------------------------------------------------------------
# Configuración del experimento (todo lo que afecta el resultado vive aquí)
# ---------------------------------------------------------------------------
SEMILLA = 42
PROPORCION_PRUEBA = 0.20
K_PLIEGUES = 5
METRICA_SELECCION = "roc_auc"
NOMBRE_EXPERIMENTO = "Actividad5-churn-telco"
COLORES = {"regresion_logistica": "#2a78d6", "random_forest": "#eb6834", "referencia": "#8a8984"}
ETIQUETAS = {"regresion_logistica": "Regresión logística", "random_forest": "Random forest",
             "referencia": "Referencia (clase mayoritaria)"}

ESCORERS = {
    "roc_auc": "roc_auc",
    "f1": "f1",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "average_precision": "average_precision",
}

_SK_NUEVO = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8)


def espacio_busqueda() -> dict:
    """Modelos base y rejillas de hiperparámetros.

    Regresión logística: C (inverso de la regularización) en escala logarítmica,
    tipo de regularización (L1 hace selección de variables; L2 encoge) y
    ponderación de clases para el desbalance 74/26.  24 combinaciones.
    Random forest: profundidad máxima y mínimo de muestras por hoja controlan el
    sobreajuste; max_features la decorrelación entre árboles; class_weight el
    desbalance. 300 árboles fijos (más árboles no cambian el sesgo, sólo la
    varianza y el costo).  48 combinaciones.
    """
    rejilla_lr = {"clf__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], "clf__class_weight": [None, "balanced"]}
    if _SK_NUEVO:
        rejilla_lr["clf__l1_ratio"] = [0.0, 1.0]          # 0 = L2, 1 = L1 (scikit-learn ≥ 1.8)
    else:
        rejilla_lr["clf__penalty"] = ["l2", "l1"]
    return {
        "regresion_logistica": (
            LogisticRegression(solver="liblinear", max_iter=5000, random_state=SEMILLA),
            rejilla_lr,
        ),
        "random_forest": (
            RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
            {
                "clf__max_depth": [6, 10, None],
                "clf__min_samples_leaf": [1, 10, 30, 60],
                "clf__max_features": ["sqrt", 0.3],
                "clf__class_weight": [None, "balanced_subsample"],
            },
        ),
    }


def construir_pipeline(estimador) -> Pipeline:
    """Escalado de numéricas + one-hot de categóricas + clasificador.
    Se ajusta dentro de cada pliegue, así que no hay fuga de información."""
    pre = ColumnTransformer(
        [
            ("num", StandardScaler(), dp.NUMERICAS),
            ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                                  sparse_output=False), dp.CATEGORICAS),
        ],
        verbose_feature_names_out=False,
    )
    return Pipeline([("pre", pre), ("clf", estimador)])


# ---------------------------------------------------------------------------
# Datos
# ---------------------------------------------------------------------------
def cargar_particion(raiz: Path):
    """Lee el dataset limpio versionado y lo divide 80/20 estratificado."""
    ruta = raiz / "datos" / "datos_limp" / dp.NOMBRE_LIMP
    meta = json.loads((raiz / "datos" / "datos_limp" /
                       dp.NOMBRE_LIMP.replace(".csv", ".meta.json")).read_text(encoding="utf-8"))
    d = dp.cargar_limpio(ruta)
    X = d[dp.CATEGORICAS + dp.NUMERICAS]
    y = d[dp.OBJETIVO].astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=PROPORCION_PRUEBA, stratify=y, random_state=SEMILLA)
    return d, X_tr, X_te, y_tr, y_te, meta, ruta


# ---------------------------------------------------------------------------
# Métricas
# ---------------------------------------------------------------------------
def metricas(y, proba, umbral: float = 0.5) -> dict:
    pred = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "average_precision": average_precision_score(y, proba),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


def umbral_optimo_f1(y, proba) -> float:
    """Umbral que maximiza F1 sobre probabilidades fuera de pliegue (sólo entrenamiento)."""
    p, r, u = precision_recall_curve(y, proba)
    f1 = np.where(p[:-1] + r[:-1] > 0, 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1]), 0)
    return float(u[int(np.argmax(f1))])


def nombres_hiperparametros(params: dict) -> dict:
    """Quita el prefijo clf__ y normaliza l1_ratio/penalty a 'regularizacion'."""
    limpio = {}
    for k, v in params.items():
        k = k.replace("clf__", "")
        if k == "l1_ratio":
            k, v = "regularizacion", "L1" if v == 1.0 else "L2"
        elif k == "penalty":
            k, v = "regularizacion", v.upper()
        limpio[k] = "None" if v is None else v
    return limpio


# ---------------------------------------------------------------------------
# Gráficas que se adjuntan como artefactos
# ---------------------------------------------------------------------------
def _guardar(fig, ruta: Path) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(ruta, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return ruta


def grafica_confusion(y, proba, umbral, titulo, ruta) -> Path:
    cm = confusion_matrix(y, (proba >= umbral).astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "#1b1b1b", fontsize=12)
    ax.set_xticks([0, 1], ["Permanece", "Abandona"])
    ax.set_yticks([0, 1], ["Permanece", "Abandona"])
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title(titulo, fontsize=10)
    return _guardar(fig, ruta)


def grafica_roc(y, proba, titulo, ruta) -> Path:
    fpr, tpr, _ = roc_curve(y, proba)
    fig, ax = plt.subplots(figsize=(4, 3.6))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y, proba):.3f}")
    ax.plot([0, 1], [0, 1], ls="--", c="#999")
    ax.set_xlabel("Tasa de falsos positivos")
    ax.set_ylabel("Tasa de verdaderos positivos")
    ax.set_title(titulo, fontsize=10)
    ax.legend(loc="lower right", frameon=False)
    return _guardar(fig, ruta)


# ---------------------------------------------------------------------------
# Entrenamiento con registro en MLflow
# ---------------------------------------------------------------------------
def configurar_mlflow(raiz: Path, tracking_uri: str | None = None) -> str:
    uri = tracking_uri or os.environ.get("MLFLOW_TRACKING_URI") or f"sqlite:///{(raiz / 'mlflow.db').as_posix()}"
    mlflow.set_tracking_uri(uri)
    if mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO) is None:
        mlflow.create_experiment(
            NOMBRE_EXPERIMENTO,
            artifact_location=(raiz / "mlartifacts").resolve().as_uri(),
            tags={"proyecto": "MSCA2102 Actividad 5", "problema": "clasificacion binaria",
                  "dataset": "IBM Telco Customer Churn",
                  # Hace que la interfaz de MLflow 3 abra el experimento en modo "Model training"
                  "mlflow.experimentKind": "custom_model_development"},
        )
    mlflow.set_experiment(NOMBRE_EXPERIMENTO)
    return uri


def _commit_git(raiz: Path) -> str:
    try:
        return subprocess.check_output(["git", "-C", str(raiz), "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return "sin-git"


def etiquetas_comunes(raiz: Path, meta: dict) -> dict:
    return {
        "dataset_version": meta["version"],
        "dataset_sha256": meta["sha256_limpio"][:16],
        "dataset_ini_sha256": meta["sha256_ini"][:16],
        "git_commit": _commit_git(raiz),
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "mlflow": mlflow.__version__,
        "semilla": str(SEMILLA),
    }


def registrar_dataset(d: pd.DataFrame, ruta: Path, meta: dict) -> None:
    ds = mlflow.data.from_pandas(d, source=str(ruta), targets=dp.OBJETIVO,
                                 name=f"telco_churn_limpio_v{meta['version']}")
    mlflow.log_input(ds, context="entrenamiento")


def entrenar_referencia(X_tr, X_te, y_tr, y_te, tags, carpeta: Path) -> dict:
    """Línea base: predice siempre la clase mayoritaria (piso que todo modelo debe superar)."""
    with mlflow.start_run(run_name="referencia_clase_mayoritaria") as run:
        mlflow.set_tags({**tags, "tipo_run": "referencia", "modelo": "DummyClassifier"})
        modelo = construir_pipeline(DummyClassifier(strategy="most_frequent"))
        cv = StratifiedKFold(K_PLIEGUES, shuffle=True, random_state=SEMILLA)
        res = cross_validate(modelo, X_tr, y_tr, cv=cv, scoring=ESCORERS)
        modelo.fit(X_tr, y_tr)
        proba = modelo.predict_proba(X_te)[:, 1]
        m_te = metricas(y_te, proba)
        mlflow.log_params({"estrategia": "most_frequent", "k_pliegues": K_PLIEGUES})
        mlflow.log_metrics({f"cv_{k}_media": float(np.mean(res[f"test_{k}"])) for k in ESCORERS})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in m_te.items()})
        return {"run_id": run.info.run_id, "test": m_te, "proba": proba}


def entrenar_con_grid(nombre: str, estimador, rejilla: dict, X_tr, X_te, y_tr, y_te,
                      d: pd.DataFrame, ruta_datos: Path, meta: dict, tags: dict,
                      carpeta: Path, registrar_modelo: bool = True) -> dict:
    """GridSearchCV (k-fold estratificado) dentro de un run padre; cada candidato
    del grid se registra como run hijo con sus parámetros y métricas de CV."""
    cv = StratifiedKFold(K_PLIEGUES, shuffle=True, random_state=SEMILLA)
    grid = GridSearchCV(construir_pipeline(estimador), rejilla, scoring=ESCORERS,
                        refit=METRICA_SELECCION, cv=cv, n_jobs=-1, return_train_score=True)
    with mlflow.start_run(run_name=f"{nombre}_gridsearch") as padre:
        mlflow.set_tags({**tags, "tipo_run": "gridsearch", "modelo": type(estimador).__name__,
                         "metrica_seleccion": METRICA_SELECCION})
        registrar_dataset(d, ruta_datos, meta)
        grid.fit(X_tr, y_tr)
        cvr = pd.DataFrame(grid.cv_results_)

        # --- runs hijos: un candidato del grid por run ---
        for i, fila in cvr.iterrows():
            with mlflow.start_run(run_name=f"{nombre}_cand_{i:02d}", nested=True):
                mlflow.set_tags({"tipo_run": "candidato", "modelo": type(estimador).__name__,
                                 "rank_roc_auc": int(fila["rank_test_roc_auc"])})
                mlflow.log_params(nombres_hiperparametros(fila["params"]))
                mlflow.log_metrics({f"cv_{k}_media": float(fila[f"mean_test_{k}"]) for k in ESCORERS})
                mlflow.log_metrics({f"cv_{k}_desv": float(fila[f"std_test_{k}"]) for k in ESCORERS})
                mlflow.log_metric("cv_roc_auc_train_media", float(fila["mean_train_roc_auc"]))
                mlflow.log_metric("tiempo_ajuste_s", float(fila["mean_fit_time"]))

        mejor = grid.best_estimator_
        i_mejor = grid.best_index_
        # Umbral que maximiza F1 con probabilidades fuera de pliegue (sin tocar prueba)
        proba_oof = cross_val_predict(clone(mejor), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
        umbral = umbral_optimo_f1(y_tr, proba_oof)
        proba_te = mejor.predict_proba(X_te)[:, 1]
        m_05 = metricas(y_te, proba_te, 0.5)
        m_u = metricas(y_te, proba_te, umbral)

        mlflow.log_params({
            **{f"mejor_{k}": v for k, v in nombres_hiperparametros(grid.best_params_).items()},
            "k_pliegues": K_PLIEGUES, "n_candidatos": len(cvr), "proporcion_prueba": PROPORCION_PRUEBA,
            "metrica_seleccion": METRICA_SELECCION, "umbral_f1_oof": round(umbral, 4),
            "escalado": "StandardScaler", "codificacion": "OneHotEncoder(drop=if_binary)",
            "n_entrenamiento": len(X_tr), "n_prueba": len(X_te),
            **{f"base_{k}": v for k, v in estimador.get_params().items()
               if k in ("solver", "max_iter", "n_estimators", "random_state")},
        })
        mlflow.log_metrics({f"cv_{k}_media": float(cvr.loc[i_mejor, f"mean_test_{k}"]) for k in ESCORERS})
        mlflow.log_metrics({f"cv_{k}_desv": float(cvr.loc[i_mejor, f"std_test_{k}"]) for k in ESCORERS})
        mlflow.log_metric("cv_roc_auc_train_media", float(cvr.loc[i_mejor, "mean_train_roc_auc"]))
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in m_05.items()})
        mlflow.log_metrics({f"test_umbral_{k}": float(v) for k, v in m_u.items()
                            if k in ("accuracy", "precision", "recall", "f1")})

        # --- artefactos ---
        sub = carpeta / nombre
        sub.mkdir(parents=True, exist_ok=True)
        cvr_out = cvr.drop(columns=[c for c in cvr.columns if c.startswith("split")])
        cvr_out.to_csv(sub / "cv_results.csv", index=False)
        grafica_confusion(y_te, proba_te, 0.5, f"{ETIQUETAS[nombre]} · umbral 0.50", sub / "confusion_050.png")
        grafica_confusion(y_te, proba_te, umbral, f"{ETIQUETAS[nombre]} · umbral {umbral:.2f}",
                          sub / "confusion_umbral.png")
        grafica_roc(y_te, proba_te, f"ROC en prueba · {ETIQUETAS[nombre]}", sub / "roc_prueba.png")
        (sub / "metricas_prueba.json").write_text(json.dumps(
            {"umbral_050": m_05, "umbral_f1_oof": {"umbral": umbral, **m_u}}, indent=2), encoding="utf-8")
        mlflow.log_artifacts(str(sub), artifact_path="evaluacion")

        firma = infer_signature(X_tr.astype({c: str for c in dp.CATEGORICAS}), proba_te[:5])
        ejemplo = X_tr.head(3).astype({c: str for c in dp.CATEGORICAS})
        extra = {}
        if "skops_trusted_types" in inspect.signature(mlflow.sklearn.log_model).parameters:
            # MLflow ≥ 3.x serializa con skops: se declaran los tipos propios del modelo
            # (creado por nosotros mismos, por eso es seguro confiar en ellos).
            extra["skops_trusted_types"] = ["sklearn.tree._tree.Tree"]
        info = mlflow.sklearn.log_model(
            mejor, name="modelo", signature=firma, input_example=ejemplo,
            registered_model_name=f"churn_{nombre}" if registrar_modelo else None, **extra,
        )
        return {"run_id": padre.info.run_id, "grid": grid, "cv_results": cvr, "mejor": mejor,
                "umbral": umbral, "proba_te": proba_te, "proba_oof": proba_oof,
                "test_050": m_05, "test_umbral": m_u, "model_uri": info.model_uri}


# ---------------------------------------------------------------------------
# Comparación estadística
# ---------------------------------------------------------------------------
def ttest_corregido(diferencias: np.ndarray, n_train: int, n_test: int) -> tuple[float, float]:
    """t de remuestreo corregida (Nadeau y Bengio, 2003): corrige la varianza por el
    traslape entre conjuntos de entrenamiento de distintos pliegues."""
    k = len(diferencias)
    media, var = diferencias.mean(), diferencias.var(ddof=1)
    t = media / np.sqrt((1 / k + n_test / n_train) * var)
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return float(t), float(p)


def comparar_modelos(res_a: dict, res_b: dict, X_tr, y_tr, y_te, tags: dict, carpeta: Path,
                     n_bootstrap: int = 2000) -> dict:
    """Compara los dos mejores modelos con: (1) CV repetida 10×3 sobre los mismos
    pliegues + t corregida y Wilcoxon; (2) bootstrap pareado del AUC en prueba;
    (3) McNemar sobre las predicciones en prueba con el umbral de cada modelo."""
    rcv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=SEMILLA)
    pliegues = {}
    for nombre, res in (("regresion_logistica", res_a), ("random_forest", res_b)):
        r = cross_validate(clone(res["mejor"]), X_tr, y_tr, cv=rcv,
                           scoring={"roc_auc": "roc_auc", "average_precision": "average_precision", "f1": "f1"},
                           n_jobs=-1)
        pliegues[nombre] = r
    n_test_f = len(y_tr) // 10
    n_train_f = len(y_tr) - n_test_f
    filas = []
    for met in ("roc_auc", "average_precision", "f1"):
        a = pliegues["regresion_logistica"][f"test_{met}"]
        b = pliegues["random_forest"][f"test_{met}"]
        dif = a - b
        t, p_t = ttest_corregido(dif, n_train_f, n_test_f)
        w = stats.wilcoxon(a, b)
        filas.append({"metrica": met, "media_lr": a.mean(), "desv_lr": a.std(ddof=1),
                      "media_rf": b.mean(), "desv_rf": b.std(ddof=1), "dif_media": dif.mean(),
                      "lr_gana_pliegues": int((dif > 0).sum()), "n_pliegues": len(dif),
                      "t_corregida": t, "p_t_corregida": p_t,
                      "wilcoxon_W": float(w.statistic), "p_wilcoxon": float(w.pvalue)})
    tabla_cv = pd.DataFrame(filas)

    # Bootstrap pareado del AUC en prueba
    rng = np.random.default_rng(SEMILLA)
    ya = np.asarray(y_te)
    pa, pb = res_a["proba_te"], res_b["proba_te"]
    difs, aucs_a, aucs_b = [], [], []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(ya), len(ya))
        if ya[idx].min() == ya[idx].max():
            continue
        aa, bb = roc_auc_score(ya[idx], pa[idx]), roc_auc_score(ya[idx], pb[idx])
        aucs_a.append(aa); aucs_b.append(bb); difs.append(aa - bb)
    difs = np.array(difs)
    boot = {
        "auc_lr": roc_auc_score(ya, pa), "auc_lr_ic95": np.percentile(aucs_a, [2.5, 97.5]).tolist(),
        "auc_rf": roc_auc_score(ya, pb), "auc_rf_ic95": np.percentile(aucs_b, [2.5, 97.5]).tolist(),
        "dif_auc": float(roc_auc_score(ya, pa) - roc_auc_score(ya, pb)),
        "dif_auc_ic95": np.percentile(difs, [2.5, 97.5]).tolist(),
        "p_bootstrap_bilateral": float(min(1.0, 2 * min((difs <= 0).mean(), (difs >= 0).mean()))),
        "n_bootstrap": int(len(difs)),
    }

    # McNemar (con corrección de continuidad) sobre aciertos en prueba
    pred_a = (pa >= res_a["umbral"]).astype(int)
    pred_b = (pb >= res_b["umbral"]).astype(int)
    ok_a, ok_b = pred_a == ya, pred_b == ya
    n01, n10 = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10) if (n01 + n10) else 0.0
    mcnemar = {"lr_acierta_rf_falla": n01, "rf_acierta_lr_falla": n10,
               "chi2": float(chi2), "p_mcnemar": float(stats.chi2.sf(chi2, 1))}

    # Gráfica de distribución por pliegue
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.3))
    for ax, met, titulo in zip(axes, ("roc_auc", "average_precision", "f1"),
                               ("ROC-AUC", "Precisión promedio (PR-AUC)", "F1 (umbral 0.50)")):
        datos = [pliegues["regresion_logistica"][f"test_{met}"], pliegues["random_forest"][f"test_{met}"]]
        bp = ax.boxplot(datos, widths=0.5, patch_artist=True)
        bp["boxes"][0].set_facecolor(COLORES["regresion_logistica"] + "55")
        bp["boxes"][1].set_facecolor(COLORES["random_forest"] + "55")
        for k, serie in enumerate(datos, start=1):
            ax.scatter(np.full(len(serie), k) + np.random.default_rng(k).uniform(-.12, .12, len(serie)),
                       serie, s=9, c=list(COLORES.values())[k - 1], alpha=.7, zorder=3)
        ax.set_xticks([1, 2], ["Reg. logística", "Random forest"])
        ax.set_title(f"{titulo} · 30 pliegues", fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#e6e5e1", lw=0.6)
    sub = carpeta / "comparacion"
    sub.mkdir(parents=True, exist_ok=True)
    _guardar(fig, sub / "pliegues_cv_repetida.png")
    tabla_cv.to_csv(sub / "pruebas_cv_repetida.csv", index=False)
    pd.DataFrame({"lr_roc_auc": pliegues["regresion_logistica"]["test_roc_auc"],
                  "rf_roc_auc": pliegues["random_forest"]["test_roc_auc"],
                  "lr_average_precision": pliegues["regresion_logistica"]["test_average_precision"],
                  "rf_average_precision": pliegues["random_forest"]["test_average_precision"],
                  "lr_f1": pliegues["regresion_logistica"]["test_f1"],
                  "rf_f1": pliegues["random_forest"]["test_f1"]}).to_csv(sub / "pliegues.csv", index=False)
    (sub / "bootstrap_mcnemar.json").write_text(json.dumps({"bootstrap_auc": boot, "mcnemar": mcnemar},
                                                           indent=2), encoding="utf-8")

    with mlflow.start_run(run_name="comparacion_estadistica"):
        mlflow.set_tags({**tags, "tipo_run": "comparacion",
                         "run_lr": res_a["run_id"], "run_rf": res_b["run_id"]})
        mlflow.log_params({"cv_repetida": "RepeatedStratifiedKFold(10x3)", "n_bootstrap": n_bootstrap,
                           "prueba_pareada": "t corregida Nadeau-Bengio + Wilcoxon"})
        for _, f in tabla_cv.iterrows():
            m = f["metrica"]
            mlflow.log_metrics({f"{m}_dif_media": f["dif_media"], f"{m}_p_t_corregida": f["p_t_corregida"],
                                f"{m}_p_wilcoxon": f["p_wilcoxon"]})
        mlflow.log_metrics({"auc_test_dif": boot["dif_auc"], "auc_test_dif_ic_inf": boot["dif_auc_ic95"][0],
                            "auc_test_dif_ic_sup": boot["dif_auc_ic95"][1],
                            "p_bootstrap": boot["p_bootstrap_bilateral"], "p_mcnemar": mcnemar["p_mcnemar"]})
        mlflow.log_artifacts(str(sub), artifact_path="comparacion")
    return {"tabla_cv": tabla_cv, "pliegues": pliegues, "bootstrap": boot, "mcnemar": mcnemar}


# ---------------------------------------------------------------------------
# Flujo completo
# ---------------------------------------------------------------------------
def ejecutar(raiz: str | Path = ".", tracking_uri: str | None = None) -> dict:
    raiz = Path(raiz).resolve()
    np.random.seed(SEMILLA)
    uri = configurar_mlflow(raiz, tracking_uri)
    d, X_tr, X_te, y_tr, y_te, meta, ruta = cargar_particion(raiz)
    tags = etiquetas_comunes(raiz, meta)
    carpeta = raiz / "resultados"
    salida = {"tracking_uri": uri, "X_tr": X_tr, "X_te": X_te, "y_tr": y_tr, "y_te": y_te, "datos": d}
    salida["referencia"] = entrenar_referencia(X_tr, X_te, y_tr, y_te, tags, carpeta)
    for nombre, (est, rejilla) in espacio_busqueda().items():
        print(f"GridSearchCV · {nombre} · {np.prod([len(v) for v in rejilla.values()])} candidatos × {K_PLIEGUES} pliegues")
        salida[nombre] = entrenar_con_grid(nombre, est, rejilla, X_tr, X_te, y_tr, y_te,
                                           d, ruta, meta, tags, carpeta)
    salida["comparacion"] = comparar_modelos(salida["regresion_logistica"], salida["random_forest"],
                                             X_tr, y_tr, y_te, tags, carpeta)
    # Exporta la tabla de runs (respaldo legible de lo registrado en MLflow)
    exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
    runs = mlflow.search_runs([exp.experiment_id], order_by=["attributes.start_time ASC"])
    runs.to_csv(carpeta / "mlflow_runs.csv", index=False)
    salida["runs"] = runs
    return salida


if __name__ == "__main__":
    s = ejecutar(Path(__file__).resolve().parents[1])
    for n in ("regresion_logistica", "random_forest"):
        r = s[n]
        print(f"{n:<22} mejores={nombres_hiperparametros(r['grid'].best_params_)}  "
              f"AUC cv={r['grid'].best_score_:.4f}  AUC prueba={r['test_050']['roc_auc']:.4f}  "
              f"F1 prueba={r['test_050']['f1']:.4f}")
    print(s["comparacion"]["tabla_cv"].round(4).to_string(index=False))
    print(json.dumps(s["comparacion"]["bootstrap"], indent=1))
    print(s["comparacion"]["mcnemar"])
    print(f"Runs registrados en {s['tracking_uri']}: {len(s['runs'])}")

In [ ]:
import importlib
sys.path.insert(0, str(RAIZ / "fuentes"))
import datos_prep as dp
import train as tr
importlib.reload(dp); importlib.reload(tr)

import numpy as np, pandas as pd, sklearn, mlflow
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

entorno = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
           "scikit-learn": sklearn.__version__, "mlflow": mlflow.__version__,
           "plataforma": "Google Colab" if EN_COLAB else platform.platform()}
(RAIZ / "resultados" / "entorno.json").write_text(json.dumps(entorno, indent=2))
pd.Series(entorno)

In [ ]:
# Estilo de las gráficas
AZUL, NARANJA, AQUA, AMARILLO, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
TINTA, TINTA2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.titleweight": "bold", "axes.edgecolor": "#b9b8b3", "axes.labelcolor": TINTA2,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "legend.frameon": False, "figure.facecolor": "white",
})
FIG = RAIZ / "resultados" / "figuras"
def guardar(fig, nombre):
    fig.savefig(FIG / nombre, bbox_inches="tight")
    plt.show()

## 2. Gestión y preparación de datos
### 2.1 Obtención y versión original
Se descarga el CSV publicado por IBM (si no está ya en `datos/datos_ini/`) y se calcula su huella SHA-256,
que identifica de forma única la versión 1.0.0 (datos originales sin modificar).

In [ ]:
ruta_ini = dp.descargar_datos(RAIZ / "datos" / "datos_ini" / dp.NOMBRE_INI)
crudo = dp.cargar_datos_ini(ruta_ini)
print(f"Fuente: {dp.URL_FUENTE}")
print(f"Archivo: {ruta_ini.relative_to(RAIZ)} · {crudo.shape[0]:,} filas × {crudo.shape[1]} columnas")
print(f"SHA-256 (v1.0.0): {dp.sha256(ruta_ini)}")
crudo.head()

### 2.2 Diagnóstico de calidad de los datos originales

In [ ]:
perfil = dp.perfil_calidad(crudo)
display(pd.Series(perfil, name="valor").to_frame())
tipos = pd.DataFrame({"tipo_leido": crudo.dtypes.astype(str), "valores_distintos": crudo.nunique(),
                      "ejemplo": crudo.iloc[0]})
tipos

In [ ]:
# Detalle de los problemas encontrados
blancos = crudo[crudo["TotalCharges"].str.strip() == ""]
print("Registros con TotalCharges en blanco:", len(blancos), "· tenure de esos registros:", sorted(blancos["tenure"].unique()))
sin_id = crudo.drop(columns=[dp.ID])
dup = sin_id[sin_id.duplicated(keep=False)]
print("Registros involucrados en duplicados exactos (sin customerID):", len(dup))
atributos = sin_id.drop(columns=[dp.OBJETIVO])
conflicto = sin_id[atributos.duplicated(keep=False)]
conflicto = conflicto.groupby(list(atributos.columns), observed=True)[dp.OBJETIVO].nunique()
print("Perfiles idénticos con etiqueta distinta (ruido de etiqueta):", int((conflicto > 1).sum()))
print("Tasa de abandono:", f"{(crudo[dp.OBJETIVO] == 'Yes').mean():.2%}")

### 2.3 Limpieza, estandarización y versión limpia (v1.1.0)
`datos_prep.preparar()` aplica las reglas R1–R11 (documentadas en el código), valida el resultado con pruebas
automáticas y guarda el CSV limpio junto con un `.meta.json` que contiene la versión, las huellas SHA-256 de los
archivos de origen y destino, y la bitácora de cada regla.

In [ ]:
meta = dp.preparar(RAIZ)
limpio = dp.cargar_limpio(RAIZ / meta["archivo_limpio"])
print(f"Dataset limpio v{meta['version']} · {meta['filas']:,} filas · abandono {meta['tasa_churn']:.2%}")
print("SHA-256 limpio:", meta["sha256_limpio"])
pd.DataFrame(meta["bitacora_limpieza"])

In [ ]:
dp.validar_limpio(limpio)
print("Validaciones superadas: sin nulos, sin duplicados, objetivo binario, rangos válidos, sin categorías residuales.")
display(limpio.dtypes.astype(str).to_frame("tipo").T)
limpio.describe().T.round(2)

In [ ]:
# Atípicos por rango intercuartílico (IQR) en las variables numéricas
filas = []
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1, q3 = limpio[c].quantile([.25, .75]); iqr = q3 - q1
    n = int(((limpio[c] < q1 - 1.5 * iqr) | (limpio[c] > q3 + 1.5 * iqr)).sum())
    filas.append({"variable": c, "Q1": q1, "Q3": q3, "IQR": iqr, "atipicos_IQR": n,
                  "asimetria": limpio[c].skew()})
pd.DataFrame(filas).round(2)

### 2.4 Visualizaciones: patrones y anomalías

In [ ]:
# Tasa de abandono por tipo de contrato, servicio de internet y método de pago
tasa_global = limpio[dp.OBJETIVO].mean()
nombres = {"Month-to-month": "Mes a mes", "One year": "Un año", "Two year": "Dos años",
           "Fiber optic": "Fibra óptica", "DSL": "DSL", "No": "Sin internet",
           "Electronic check": "Cheque electrónico", "Mailed check": "Cheque por correo",
           "Bank transfer (automatic)": "Transferencia (aut.)", "Credit card (automatic)": "Tarjeta (aut.)"}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
for ax, col, titulo in zip(axes, ["Contract", "InternetService", "PaymentMethod"],
                           ["Tipo de contrato", "Servicio de internet", "Método de pago"]):
    t = limpio.groupby(col, observed=True)[dp.OBJETIVO].agg(["mean", "size"]).sort_values("mean")
    y = [nombres.get(i, i) for i in t.index]
    ax.barh(y, t["mean"], color=NARANJA, height=0.55)
    for yi, (m, n) in enumerate(zip(t["mean"], t["size"])):
        ax.text(m + 0.01, yi, f"{m:.0%}  (n={n:,})", va="center", fontsize=8.5, color=TINTA)
    ax.axvline(tasa_global, ls="--", lw=1, color=GRIS)
    ax.set_xlim(0, 0.65); ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title(titulo); ax.grid(axis="y", visible=False)
fig.suptitle(f"Tasa de abandono por segmento (línea punteada: promedio {tasa_global:.1%})", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "eda_tasa_abandono_segmentos.png")

In [ ]:
# Distribución de las variables numéricas según abandono
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
for ax, col, titulo in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"],
                           ["Antigüedad (meses)", "Cargo mensual (USD)", "Cargo total (USD)"]):
    bins = np.histogram_bin_edges(limpio[col], bins=30)
    for val, color, etiqueta in [(0, AZUL, "Permanece"), (1, NARANJA, "Abandona")]:
        ax.hist(limpio.loc[limpio[dp.OBJETIVO] == val, col], bins=bins, density=True,
                histtype="step", lw=2, color=color, label=etiqueta)
    ax.set_title(titulo); ax.set_yticks([]); ax.grid(axis="y", visible=False)
axes[0].legend(loc="upper center")
fig.suptitle("Distribución por grupo (densidad)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "eda_distribuciones_numericas.png")

In [ ]:
# Diagramas de caja (atípicos visuales) por grupo
fig, axes = plt.subplots(1, 3, figsize=(11, 2.9))
for ax, col, titulo in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"],
                           ["Antigüedad (meses)", "Cargo mensual (USD)", "Cargo total (USD)"]):
    bp = ax.boxplot([limpio.loc[limpio[dp.OBJETIVO] == v, col] for v in (0, 1)], vert=False,
                    widths=0.5, patch_artist=True, flierprops={"marker": "o", "ms": 3, "alpha": .4})
    for caja, color in zip(bp["boxes"], [AZUL, NARANJA]):
        caja.set_facecolor(color + "44"); caja.set_edgecolor(color)
    ax.set_yticks([1, 2], ["Permanece", "Abandona"]); ax.set_title(titulo); ax.grid(axis="y", visible=False)
fig.tight_layout()
guardar(fig, "eda_cajas_numericas.png")

In [ ]:
# Abandono por antigüedad y tipo de contrato
cortes = [-1, 6, 12, 24, 48, 72]
etiq = ["0–6", "7–12", "13–24", "25–48", "49–72"]
tmp = limpio.assign(tramo=pd.cut(limpio["tenure"], cortes, labels=etiq))
tab = tmp.groupby(["Contract", "tramo"], observed=False)[dp.OBJETIVO].mean().unstack(0)
fig, ax = plt.subplots(figsize=(6.4, 3.3))
for col, color in zip(["Month-to-month", "One year", "Two year"], [NARANJA, AZUL, AQUA]):
    ax.plot(etiq, tab[col], marker="o", ms=6, lw=2, color=color, label=nombres[col])
    ax.text(len(etiq) - 0.9, tab[col].iloc[-1], nombres[col], va="center", fontsize=8.5, color=TINTA)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("Antigüedad (meses)"); ax.set_ylabel("Tasa de abandono")
ax.set_xlim(-0.3, len(etiq) - 0.2 + 0.9); ax.legend(loc="upper right")
ax.set_title("Abandono por antigüedad y contrato")
guardar(fig, "eda_abandono_antiguedad_contrato.png")
tab.round(3)

In [ ]:
# Correlaciones entre numéricas y objetivo (Pearson; con el objetivo binario es la punto-biserial)
nums = ["tenure", "MonthlyCharges", "TotalCharges", "n_servicios", dp.OBJETIVO]
corr = limpio[nums].corr()
cmap = mcolors.LinearSegmentedColormap.from_list("div", [AZUL, "#f0efec", NARANJA])
fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
etq = ["Antigüedad", "Cargo mensual", "Cargo total", "N.º servicios", "Abandono"]
ax.set_xticks(range(5), etq, rotation=35, ha="right"); ax.set_yticks(range(5), etq); ax.grid(False)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8.5, color=TINTA)
fig.colorbar(im, ax=ax, shrink=.8)
ax.set_title("Matriz de correlación")
guardar(fig, "eda_correlaciones.png")
corr.round(3)

In [ ]:
# Anomalías de consistencia: razón TotalCharges / (tenure × MonthlyCharges) frente a la antigüedad
esperado = limpio["tenure"] * limpio["MonthlyCharges"]
razon = np.where(esperado > 0, limpio["TotalCharges"] / esperado.replace(0, np.nan), np.nan)
an = limpio["anomalia_cargos"] == 1
fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.axhspan(1 - dp.TOLERANCIA_CARGOS, 1 + dp.TOLERANCIA_CARGOS, color="#f0efec", zorder=0, label="Tolerancia ±25 %")
ax.scatter(limpio.loc[~an, "tenure"], razon[~an], s=6, alpha=.3, color=AZUL, label="Consistente")
ax.scatter(limpio.loc[an, "tenure"], razon[an], s=30, color=NARANJA, edgecolor="white", lw=.6,
           label=f"Fuera de tolerancia (n={an.sum()})")
ax.axhline(1, ls="--", lw=1, color=GRIS)
ax.set_xlabel("Antigüedad (meses)"); ax.set_ylabel("TotalCharges / (tenure × MonthlyCharges)")
ax.set_title("Consistencia del cargo total"); ax.legend(loc="upper right", fontsize=8)
guardar(fig, "eda_anomalias_cargos.png")
print("Registros con tenure = 0 (TotalCharges imputado a 0 por regla):", int((limpio["tenure"] == 0).sum()))
limpio.loc[an, ["tenure", "MonthlyCharges", "TotalCharges", dp.OBJETIVO]].assign(
    razon=razon[an].round(2)).describe().round(2)

## 3. Entrenamiento y ajuste de modelos con registro en MLflow
### 3.1 Configuración de MLflow y partición
Seguimiento local con base de datos SQLite (`mlflow.db`) y artefactos en `mlartifacts/`. Partición 80/20
estratificada con semilla 42; el conjunto de prueba no se usa para ninguna decisión de ajuste.

In [ ]:
uri = tr.configurar_mlflow(RAIZ)
d, X_tr, X_te, y_tr, y_te, meta, ruta_limpio = tr.cargar_particion(RAIZ)
tags = tr.etiquetas_comunes(RAIZ, meta)
CARPETA = RAIZ / "resultados"
print("Tracking URI:", uri)
print("Experimento:", tr.NOMBRE_EXPERIMENTO)
print(f"Entrenamiento: {len(X_tr):,} (abandono {y_tr.mean():.2%}) · Prueba: {len(X_te):,} (abandono {y_te.mean():.2%})")
pd.Series(tags)

### 3.2 Espacio de búsqueda de hiperparámetros

In [ ]:
filas = []
for nombre, (est, rejilla) in tr.espacio_busqueda().items():
    for k, v in rejilla.items():
        filas.append({"modelo": nombre, "hiperparametro": k.replace("clf__", ""), "valores": v})
    filas.append({"modelo": nombre, "hiperparametro": "combinaciones",
                  "valores": int(np.prod([len(v) for v in rejilla.values()]))})
pd.DataFrame(filas)

### 3.3 Línea base y búsqueda en rejilla (GridSearchCV, k = 5)

In [ ]:
import time
t0 = time.time()
res_ref = tr.entrenar_referencia(X_tr, X_te, y_tr, y_te, tags, CARPETA)
resultados = {}
for nombre, (est, rejilla) in tr.espacio_busqueda().items():
    t1 = time.time()
    resultados[nombre] = tr.entrenar_con_grid(nombre, est, rejilla, X_tr, X_te, y_tr, y_te,
                                              d, ruta_limpio, meta, tags, CARPETA)
    print(f"{nombre:<22} {time.time() - t1:6.1f} s · mejores: "
          f"{tr.nombres_hiperparametros(resultados[nombre]['grid'].best_params_)}")
print(f"Tiempo total: {time.time() - t0:.1f} s")
lr, rf = resultados["regresion_logistica"], resultados["random_forest"]

In [ ]:
# Mejores candidatos de cada rejilla (ordenados por ROC-AUC de validación cruzada)
def top(res, n=5):
    c = res["cv_results"].sort_values("rank_test_roc_auc").head(n)
    t = pd.json_normalize(c["params"].map(tr.nombres_hiperparametros))
    for m in ["roc_auc", "average_precision", "f1", "recall", "precision"]:
        t[f"cv_{m}"] = c[f"mean_test_{m}"].values
    t["cv_roc_auc_desv"] = c["std_test_roc_auc"].values
    t["train_roc_auc"] = c["mean_train_roc_auc"].values
    return t.round(4)
display(top(lr)); top(rf)

In [ ]:
# Curvas de ajuste: regresión logística (ROC-AUC de validación frente a C)
c = lr["cv_results"].copy()
c["reg"] = c["params"].map(lambda p: tr.nombres_hiperparametros(p)["regularizacion"])
c["peso"] = c["params"].map(lambda p: "balanceado" if p["clf__class_weight"] == "balanced" else "sin ponderar")
c["C"] = c["params"].map(lambda p: p["clf__C"])
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
for ax, met, titulo in zip(axes, ["roc_auc", "f1"], ["ROC-AUC (validación)", "F1 (validación, umbral 0.50)"]):
    for (reg, peso), color, ls in zip([("L2", "sin ponderar"), ("L1", "sin ponderar"),
                                       ("L2", "balanceado"), ("L1", "balanceado")],
                                      [AZUL, AZUL, NARANJA, NARANJA], ["-", ":", "-", ":"]):
        s = c[(c.reg == reg) & (c.peso == peso)].sort_values("C")
        ax.plot(s["C"], s[f"mean_test_{met}"], marker="o", ms=5, lw=2, ls=ls, color=color,
                label=f"{reg} · {peso}")
    ax.set_xscale("log"); ax.set_xlabel("C (inverso de la regularización)"); ax.set_title(titulo)
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Regresión logística · búsqueda en rejilla", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "mod_rejilla_regresion_logistica.png")

In [ ]:
# Random forest: mapa de calor de ROC-AUC (mejor valor por profundidad × hojas) y brecha de sobreajuste
c = rf["cv_results"].copy()
for k in ["max_depth", "min_samples_leaf", "max_features", "class_weight"]:
    c[k] = c["params"].map(lambda p: str(p[f"clf__{k}"]))
piv = c.pivot_table(index="max_depth", columns="min_samples_leaf", values="mean_test_roc_auc", aggfunc="max")
piv = piv.loc[["6", "10", "None"], ["1", "10", "30", "60"]]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1.15]})
cmap_seq = mcolors.LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#2a78d6", "#0d366b"])
vmin = np.sort(piv.values.ravel())[1] - 0.002
im = axes[0].imshow(piv.values, cmap=cmap_seq, aspect="auto", vmin=vmin, vmax=piv.values.max())
axes[0].set_xticks(range(4), piv.columns); axes[0].set_yticks(range(3), ["6", "10", "Sin límite"])
axes[0].set_xlabel("min_samples_leaf"); axes[0].set_ylabel("max_depth"); axes[0].grid(False)
for i in range(3):
    for j in range(4):
        v = piv.values[i, j]
        axes[0].text(j, i, f"{v:.4f}", ha="center", va="center", fontsize=8.5,
                     color="white" if v > (vmin + piv.values.max()) / 2 else TINTA)
axes[0].set_title("ROC-AUC de validación (máximo por celda)")
g = c.groupby("min_samples_leaf")[["mean_train_roc_auc", "mean_test_roc_auc"]].mean().loc[["1", "10", "30", "60"]]
axes[1].plot(g.index, g["mean_train_roc_auc"], marker="o", lw=2, color=NARANJA, label="Entrenamiento")
axes[1].plot(g.index, g["mean_test_roc_auc"], marker="o", lw=2, color=AZUL, label="Validación")
for x, (a, b) in enumerate(zip(g["mean_train_roc_auc"], g["mean_test_roc_auc"])):
    axes[1].text(x + 0.08, (a + b) / 2, f"brecha {a - b:.3f}", fontsize=8, color=TINTA2)
axes[1].set_xlabel("min_samples_leaf"); axes[1].set_ylabel("ROC-AUC promedio"); axes[1].legend()
axes[1].set_title("Sobreajuste según tamaño mínimo de hoja")
fig.suptitle("Random forest · búsqueda en rejilla", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "mod_rejilla_random_forest.png")

### 3.4 Evaluación en el conjunto de prueba

In [ ]:
filas = []
for nombre, res in [("Referencia", None), ("Regresión logística", lr), ("Random forest", rf)]:
    if res is None:
        m = res_ref["test"]; filas.append({"modelo": nombre, "umbral": 0.5, **m}); continue
    filas.append({"modelo": nombre, "umbral": 0.5, **res["test_050"]})
    filas.append({"modelo": nombre, "umbral": round(res["umbral"], 3), **res["test_umbral"]})
tabla_prueba = pd.DataFrame(filas)
cols = ["modelo", "umbral", "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "tp", "fp", "fn", "tn"]
tabla_prueba = tabla_prueba[cols]
tabla_prueba.to_csv(CARPETA / "tabla_metricas_prueba.csv", index=False)
tabla_cv = pd.DataFrame([
    {"modelo": n, **{f"cv_{m}": r["cv_results"].loc[r["grid"].best_index_, f"mean_test_{m}"] for m in tr.ESCORERS},
     "cv_roc_auc_desv": r["cv_results"].loc[r["grid"].best_index_, "std_test_roc_auc"],
     "train_roc_auc": r["cv_results"].loc[r["grid"].best_index_, "mean_train_roc_auc"]}
    for n, r in [("Regresión logística", lr), ("Random forest", rf)]])
tabla_cv.to_csv(CARPETA / "tabla_metricas_cv.csv", index=False)
display(tabla_cv.round(4)); tabla_prueba.round(4)

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
for nombre, proba, color in [("Regresión logística", lr["proba_te"], AZUL), ("Random forest", rf["proba_te"], NARANJA)]:
    fpr, tpr, _ = roc_curve(y_te, proba)
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f"{nombre} · AUC {roc_auc_score(y_te, proba):.3f}")
    p, r, _ = precision_recall_curve(y_te, proba)
    axes[1].plot(r, p, lw=2, color=color, label=f"{nombre} · AP {average_precision_score(y_te, proba):.3f}")
axes[0].plot([0, 1], [0, 1], ls="--", lw=1, color=GRIS, label="Azar · AUC 0.500")
axes[1].axhline(y_te.mean(), ls="--", lw=1, color=GRIS, label=f"Prevalencia · {y_te.mean():.3f}")
axes[0].set_xlabel("Tasa de falsos positivos"); axes[0].set_ylabel("Tasa de verdaderos positivos")
axes[1].set_xlabel("Recall (sensibilidad)"); axes[1].set_ylabel("Precisión")
axes[0].set_title("Curva ROC (prueba)"); axes[1].set_title("Curva precisión-recall (prueba)")
for ax in axes: ax.legend(fontsize=8, loc="lower right" if ax is axes[0] else "upper right")
fig.tight_layout()
guardar(fig, "mod_curvas_roc_pr.png")

In [ ]:
from sklearn.metrics import confusion_matrix
fig, axes = plt.subplots(1, 4, figsize=(12, 2.9))
casos = [("Reg. logística · 0.50", lr["proba_te"], 0.5), (f"Reg. logística · {lr['umbral']:.2f}", lr["proba_te"], lr["umbral"]),
         ("Random forest · 0.50", rf["proba_te"], 0.5), (f"Random forest · {rf['umbral']:.2f}", rf["proba_te"], rf["umbral"])]
for ax, (titulo, proba, u) in zip(axes, casos):
    cm = confusion_matrix(y_te, (proba >= u).astype(int))
    ax.imshow(cm, cmap=cmap_seq, vmin=0, vmax=cm.max()); ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=11,
                    color="white" if cm[i, j] > cm.max() * .45 else TINTA)
    ax.set_xticks([0, 1], ["Permanece", "Abandona"]); ax.set_yticks([0, 1], ["Permanece", "Abandona"])
    ax.set_title(titulo, fontsize=9.5); ax.set_xlabel("Predicción")
axes[0].set_ylabel("Real")
fig.tight_layout()
guardar(fig, "mod_matrices_confusion.png")

In [ ]:
# Comparación de métricas en prueba con el umbral elegido en entrenamiento (OOF)
mets = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
etq_m = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
va = [lr["test_umbral"][m] for m in mets]; vb = [rf["test_umbral"][m] for m in mets]
x = np.arange(len(mets)); w = 0.36
fig, ax = plt.subplots(figsize=(8.4, 3.3))
ba = ax.bar(x - w / 2 - 0.01, va, w, color=AZUL, label=f"Regresión logística (umbral {lr['umbral']:.2f})")
bb = ax.bar(x + w / 2 + 0.01, vb, w, color=NARANJA, label=f"Random forest (umbral {rf['umbral']:.2f})")
for barras in (ba, bb):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012, f"{b.get_height():.3f}",
                ha="center", fontsize=7.5, color=TINTA)
ax.set_xticks(x, etq_m); ax.set_ylim(0, 1.15); ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", fontsize=8, ncol=2); ax.set_title("Métricas en el conjunto de prueba")
guardar(fig, "mod_metricas_prueba.png")

In [ ]:
# Importancia de variables por permutación (caída de ROC-AUC en prueba), comparable entre modelos
from sklearn.inspection import permutation_importance
imp = {}
for nombre, res in [("lr", lr), ("rf", rf)]:
    pi = permutation_importance(res["mejor"], X_te, y_te, scoring="roc_auc", n_repeats=10,
                                random_state=tr.SEMILLA, n_jobs=-1)
    imp[nombre] = pd.DataFrame({"media": pi.importances_mean, "desv": pi.importances_std}, index=X_te.columns)
orden = (imp["lr"]["media"] + imp["rf"]["media"]).sort_values(ascending=True).tail(10).index
fig, ax = plt.subplots(figsize=(7.2, 4))
yy = np.arange(len(orden)); h = 0.38
ax.barh(yy + h / 2 + 0.01, imp["lr"].loc[orden, "media"], h, xerr=imp["lr"].loc[orden, "desv"],
        color=AZUL, label="Regresión logística", error_kw={"lw": .8, "ecolor": TINTA2})
ax.barh(yy - h / 2 - 0.01, imp["rf"].loc[orden, "media"], h, xerr=imp["rf"].loc[orden, "desv"],
        color=NARANJA, label="Random forest", error_kw={"lw": .8, "ecolor": TINTA2})
ax.set_yticks(yy, orden); ax.grid(axis="y", visible=False)
ax.set_xlabel("Caída de ROC-AUC al permutar la variable"); ax.legend(loc="lower right")
ax.set_title("Importancia por permutación (10 variables principales)")
guardar(fig, "mod_importancia_permutacion.png")
tabla_imp = pd.concat({"lr": imp["lr"], "rf": imp["rf"]}, axis=1).sort_values(("rf", "media"), ascending=False)
tabla_imp.to_csv(CARPETA / "importancia_permutacion.csv")
tabla_imp.round(4).head(12)

In [ ]:
# Coeficientes de la regresión logística (sobre variables escaladas / codificadas)
modelo_lr = lr["mejor"]
coef = pd.Series(modelo_lr.named_steps["clf"].coef_[0],
                 index=modelo_lr.named_steps["pre"].get_feature_names_out()).sort_values()
coef.to_csv(CARPETA / "coeficientes_regresion_logistica.csv")
pd.concat([coef.head(6), coef.tail(6)]).round(3).to_frame("coeficiente (log-odds)")

## 4. Comparación estadística
Validación cruzada repetida 10 × 3 sobre los **mismos** pliegues para ambos modelos (con los hiperparámetros ya
elegidos), prueba *t* de remuestreo corregida (Nadeau y Bengio, 2003) y Wilcoxon de rangos con signo; además,
bootstrap pareado del ROC-AUC en prueba y prueba de McNemar sobre las predicciones.

In [ ]:
comp = tr.comparar_modelos(lr, rf, X_tr, y_tr, y_te, tags, CARPETA)
display(comp["tabla_cv"].round(4))
print(json.dumps(comp["bootstrap"], indent=1)); print(comp["mcnemar"])
from IPython.display import Image
Image(str(CARPETA / "comparacion" / "pliegues_cv_repetida.png"))

In [ ]:
import shutil
shutil.copy(CARPETA / "comparacion" / "pliegues_cv_repetida.png", FIG / "mod_pliegues_cv_repetida.png")
# Resumen consolidado en JSON (lo usa el reporte)
resumen = {
    "entorno": entorno, "dataset": {k: meta[k] for k in ["version", "sha256_ini", "sha256_limpio", "filas", "tasa_churn"]},
    "perfil_ini": meta["perfil_calidad_ini"], "bitacora": meta["bitacora_limpieza"],
    "particion": {"entrenamiento": len(X_tr), "prueba": len(X_te), "churn_tr": float(y_tr.mean()), "churn_te": float(y_te.mean())},
    "referencia": res_ref["test"],
    "modelos": {n: {"mejores": tr.nombres_hiperparametros(r["grid"].best_params_), "umbral": r["umbral"],
                    "cv": tabla_cv.set_index("modelo").loc[e].to_dict(), "test_050": r["test_050"],
                    "test_umbral": r["test_umbral"], "run_id": r["run_id"], "model_uri": r["model_uri"],
                    "n_candidatos": len(r["cv_results"])}
                for n, r, e in [("regresion_logistica", lr, "Regresión logística"), ("random_forest", rf, "Random forest")]},
    "comparacion": {"tabla_cv": comp["tabla_cv"].to_dict(orient="records"), "bootstrap": comp["bootstrap"],
                    "mcnemar": comp["mcnemar"]},
}
(CARPETA / "resumen_resultados.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))
print("Resumen guardado en resultados/resumen_resultados.json")

## 5. Evidencia de MLflow y reproducibilidad
### 5.1 Runs registrados en el experimento

In [ ]:
exp = mlflow.get_experiment_by_name(tr.NOMBRE_EXPERIMENTO)
runs = mlflow.search_runs([exp.experiment_id], order_by=["attributes.start_time ASC"])
runs.to_csv(CARPETA / "mlflow_runs.csv", index=False)
print(f"Experimento '{exp.name}' (id {exp.experiment_id}) · {len(runs)} runs · artefactos en {exp.artifact_location}")
display(runs["tags.tipo_run"].value_counts().to_frame("runs"))
padres = runs[runs["tags.tipo_run"].isin(["referencia", "gridsearch", "comparacion"])]
cols = ["tags.mlflow.runName", "run_id", "tags.tipo_run", "metrics.cv_roc_auc_media",
        "metrics.test_roc_auc", "metrics.test_f1", "metrics.test_umbral_f1", "tags.dataset_version", "tags.dataset_sha256"]
padres[[c for c in cols if c in padres.columns]].round(4)

In [ ]:
# Parámetros registrados en el run del mejor random forest
r = mlflow.get_run(rf["run_id"])
display(pd.Series(r.data.params, name="valor").sort_index().to_frame())
artefactos = [a.path for a in mlflow.MlflowClient().list_artifacts(rf["run_id"], "evaluacion")]
print("Artefactos de evaluación:", artefactos)

### 5.2 Model Registry y verificación de reproducibilidad

In [ ]:
cliente = mlflow.MlflowClient()
registro = []
for rm in cliente.search_registered_models():
    for v in cliente.search_model_versions(f"name='{rm.name}'"):
        registro.append({"modelo_registrado": rm.name, "version": v.version, "run_id": v.run_id, "estado": v.status})
display(pd.DataFrame(registro))

# Se recarga el modelo desde el registro y se verifica que reproduce exactamente las métricas registradas
filas = []
for nombre, res in [("churn_regresion_logistica", lr), ("churn_random_forest", rf)]:
    version = max(int(v["version"]) for v in registro if v["modelo_registrado"] == nombre)
    cargado = mlflow.sklearn.load_model(f"models:/{nombre}/{version}")
    auc = roc_auc_score(y_te, cargado.predict_proba(X_te)[:, 1])
    filas.append({"modelo": nombre, "version": version, "auc_registrado": res["test_050"]["roc_auc"],
                  "auc_recargado": auc, "identico": bool(np.isclose(auc, res["test_050"]["roc_auc"]))})
pd.DataFrame(filas)

### 5.3 Abrir el panel de MLflow
En Colab, la celda siguiente levanta la interfaz de MLflow y la abre en una pestaña del navegador
(capturas para la evidencia). En una computadora local: `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000`
desde la carpeta `Actividad5/` y abrir http://127.0.0.1:5000.

In [ ]:
if EN_COLAB:
    import subprocess, time
    subprocess.Popen(["mlflow", "ui", "--backend-store-uri", f"sqlite:///{RAIZ}/mlflow.db",
                      "--host", "127.0.0.1", "--port", "5000"], cwd=RAIZ,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(8)
    from google.colab import output
    output.serve_kernel_port_as_window(5000, path="/")
else:
    print("Ejecución local: abre una terminal en", RAIZ, "y corre  mlflow ui --backend-store-uri sqlite:///mlflow.db")

## 6. Empaquetar el proyecto

In [ ]:
import shutil
destino = shutil.make_archive(str(RAIZ.parent / "Actividad5"), "zip", RAIZ.parent, RAIZ.name)
print("Archivo generado:", destino)
if EN_COLAB:
    from google.colab import files
    files.download(destino)